In [ ]:
import pandas as pd 
import numpy as np 
import pickle 
import matplotlib.pyplot as plt

In [ ]:
key = "" # sample identifier

In [ ]:
pixel_clustering_path = "" # pixel clustering results

In [ ]:
with open(pixel_clustering_path + '/cluster_color_dict.pickle', 'rb') as f: 
    cluster_color_dict = pickle.load(f)

In [ ]:
weights_df = pd.read_csv(pixel_clustering_path + '/som_weights.csv').drop(columns=['Unnamed: 0'])

In [ ]:
non_empty_pixel_som_clusters = pd.read_csv(pixel_clustering_path + '/non_empty_pixels_som_clusters.csv').drop(columns=['Unnamed: 0'])

In [ ]:
cmap = plt.cm.colors.ListedColormap(list(cluster_color_dict.values()))

In [ ]:

def cluster_mean_contributions(X, labels):
    if isinstance(X, pd.DataFrame):
        feature_names = X.columns
        X = X.values
    else:
        feature_names = [f"feature_{i}" for i in range(X.shape[1])]
    
    labels = np.array(labels)
    unique_clusters = np.unique(labels)
    
    cluster_means = []
    for c in unique_clusters:
        cluster_mean = X[labels == c].mean(axis=0)
        cluster_means.append(cluster_mean)
    
    cluster_means = np.array(cluster_means)
    cluster_contrib = cluster_means / cluster_means.sum(axis=1, keepdims=True)
    
    df = pd.DataFrame(cluster_contrib, columns=feature_names, index=unique_clusters)
    df.index.name = "Cluster"
    return df


def plot_stacked_bar(df, xlabel, ylabel):
    plt.rcParams.update({
    "font.size": 20
    })
    ax = df.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='tab20')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend(title="Features", bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=15)
    plt.tight_layout()
    plt.show()

def analyze_som_feature_importance(X, labels):
    cluster_contrib_df = cluster_mean_contributions(X, labels)
    plot_stacked_bar(cluster_contrib_df, "Cluster", "Contribution")


analyze_som_feature_importance(non_empty_pixel_som_clusters.iloc[:, :-3], np.array(non_empty_pixel_som_clusters.iloc[:, -3]))
